# CivicBook Component 4

**XLM-R + BIO tagging, with chained institution → department classification and confidence gates**

| Part | What it does |
|---|---|
| 1 | Settings, folders, reproducibility |
| 2 | Load the dataset (`train/validation/test.jsonl`) and check it |
| 3 | Turn word tokens + BIO tags into XLM-R inputs |
| 4 | The model: XLM-R encoder + BIO head + institution head + one department head per institution |
| 5 | Training with validation after every epoch (best model kept) |
| 6 | Prediction, metrics, and confidence-threshold calibration on validation |
| 7 | Baselines: TF-IDF single-label classifier and rule-based splitter |
| 8 | Final test evaluation, comparison table, statistical tests, error files |
| 9 | Routing engine (confidence gates + `is_fake`) and a demo for PP1 |

Everything you need afterwards is saved in `/kaggle/working/outputs/`.

## Part 1: Settings

In [ ]:
import os, json, math, random, re, time, glob, unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

QUICK_RUN = False                       # True = small fast test of the whole notebook

MODEL_NAME = os.environ.get("C4_MODEL", "xlm-roberta-base")   # or "xlm-roberta-large"
MAX_LEN = 512                           # records are <= 150 words, so nothing is cut
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
EPOCHS = 6
LEARNING_RATE = 3e-5                    # encoder
HEAD_LEARNING_RATE = 1e-3               # the small classification heads
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOP_PATIENCE = 2                 # stop if validation does not improve for 2 epochs
DROPOUT = 0.1
LOSS_WEIGHTS = {"bio": 1.0, "institution": 1.0, "department": 1.0}
SPAN_MATCH_IOU = 0.5                    # predicted span counts as the same sub-complaint if overlap >= 50%
TARGET_GATE_ACCURACY = 0.95             # thresholds are chosen so confident predictions reach this accuracy
SEED = 42

if os.environ.get("C4_QUICK"):
    QUICK_RUN = os.environ["C4_QUICK"] == "1"
if os.environ.get("C4_EPOCHS"):
    EPOCHS = int(os.environ["C4_EPOCHS"])
if QUICK_RUN:
    EPOCHS = 1

# ---- folders
def find_data_dir():
    if os.environ.get("C4_DATA"):
        return Path(os.environ["C4_DATA"])
    hits = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
    if not hits:
        raise FileNotFoundError("train.jsonl not found under /kaggle/input. Add your dataset with "
                                "'+ Add Input' (see the guide) and run this cell again.")
    return Path(hits[0]).parent

DATA_DIR = find_data_dir()
OUT_DIR = Path(os.environ.get("C4_OUT", "/kaggle/working/outputs"))
for sub in ["model", "metrics", "predictions", "figures"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(SEED)

print("Data      :", DATA_DIR, sorted(os.listdir(DATA_DIR)))
print("Outputs   :", OUT_DIR)
print("Device    :", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print("Model     :", MODEL_NAME, "| quick run" if QUICK_RUN else "| full run")

## Part 2: Load the dataset and check it
The `.jsonl` files made by `build_dataset.py` are used directly.

In [ ]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

data = {split: read_jsonl(DATA_DIR / f"{split}.jsonl") for split in ["train", "validation", "test"]}

if QUICK_RUN:
    rnd = random.Random(SEED)
    data["train"] = rnd.sample(data["train"], min(600, len(data["train"])))
    data["validation"] = rnd.sample(data["validation"], min(150, len(data["validation"])))
    data["test"] = rnd.sample(data["test"], min(200, len(data["test"])))

# ---- label maps (built from the training data, so they are fixed for the model)
INSTITUTIONS = sorted({s["institution"] for r in data["train"] for s in r["sub_complaints"]})
DEPARTMENTS = {inst: sorted({s["department"] for r in data["train"] for s in r["sub_complaints"]
                              if s["institution"] == inst}) for inst in INSTITUTIONS}
INST2ID = {x: i for i, x in enumerate(INSTITUTIONS)}
DEPT2ID = {inst: {d: i for i, d in enumerate(ds)} for inst, ds in DEPARTMENTS.items()}
TAGS = ["O", "B", "I"]
TAG2ID = {t: i for i, t in enumerate(TAGS)}

# ---- checks (the dataset script already checked these; re-checked here after upload)
for split, recs in data.items():
    for r in recs:
        assert len(r["tokens"]) == len(r["bio_tags"]), r["complaint_id"]
        for s in r["sub_complaints"]:
            assert s["institution"] in INST2ID and s["department"] in DEPT2ID[s["institution"]], r["complaint_id"]
train_src = {s["source_id"] for r in data["train"] for s in r["sub_complaints"]}
test_src = {s["source_id"] for r in data["test"] for s in r["sub_complaints"]}
assert not (train_src & test_src), "LEAKAGE: a real complaint is in both train and test"

print("Institutions:", INSTITUTIONS)
print("Departments :", DEPARTMENTS)
summary = pd.DataFrame([(sp, r["complaint_type"]) for sp, rs in data.items() for r in rs],
                       columns=["split", "type"]).value_counts().unstack(0).fillna(0).astype(int)
print(summary[["train", "validation", "test"]])
print("\nAll checks passed. No real complaint is shared between train and test.")

## Part 3: From word tokens to XLM-R inputs
XLM-R cuts each word into smaller pieces. The **first piece of each word gets the word's BIO tag**; the other pieces get `-100`, which the loss ignores. Each sub-complaint's word range is turned into a piece range, used to pool (average) that sub-complaint's representation.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(record):
    enc = tokenizer(record["tokens"], is_split_into_words=True, truncation=True, max_length=MAX_LEN)
    word_ids = enc.word_ids()
    labels, prev = [], None
    for w in word_ids:
        if w is None or w == prev:
            labels.append(-100)
        else:
            labels.append(TAG2ID[record["bio_tags"][w]])
        prev = w
    # first and last piece of every word
    first, last = {}, {}
    for pos, w in enumerate(word_ids):
        if w is None:
            continue
        first.setdefault(w, pos)
        last[w] = pos
    spans = []
    for s in record["sub_complaints"]:
        ws = [w for w in range(s["token_start"], s["token_end"]) if w in first]
        if not ws:
            continue                      # cut off by MAX_LEN (should not happen)
        spans.append((first[ws[0]], last[ws[-1]] + 1,
                      INST2ID[s["institution"]], DEPT2ID[s["institution"]][s["department"]]))
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"],
            "labels": labels, "spans": spans, "first": first, "last": last,
            "n_words": len(record["tokens"])}

class ComplaintDataset(Dataset):
    def __init__(self, records):
        self.records = records
        self.items = [encode(r) for r in records]
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        return self.items[i]

def collate(batch):
    width = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    ids = torch.full((len(batch), width), pad, dtype=torch.long)
    mask = torch.zeros((len(batch), width), dtype=torch.long)
    labels = torch.full((len(batch), width), -100, dtype=torch.long)
    span_rows = []
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[i, :n] = torch.tensor(b["input_ids"])
        mask[i, :n] = torch.tensor(b["attention_mask"])
        labels[i, :n] = torch.tensor(b["labels"])
        for (s, e, inst, dept) in b["spans"]:
            span_rows.append((i, s, e, inst, dept))
    spans = torch.tensor(span_rows, dtype=torch.long) if span_rows else torch.zeros((0, 5), dtype=torch.long)
    return {"input_ids": ids, "attention_mask": mask, "labels": labels, "spans": spans}

datasets = {split: ComplaintDataset(recs) for split, recs in data.items()}
lengths = [len(x["input_ids"]) for x in datasets["train"].items]
cut = sum(len(x["spans"]) < len(r["sub_complaints"]) for x, r in zip(datasets["train"].items, data["train"]))
print(f"XLM-R pieces per training record: median {np.median(lengths):.0f}, max {max(lengths)} (limit {MAX_LEN})")
print("Records with a sub-complaint cut off by the length limit:", cut)

# show one example of the alignment
ex = data["train"][0]
enc = tokenizer(ex["tokens"], is_split_into_words=True)
print("\nExample:", ex["raw_complaint_text"][:120])
print([(tokenizer.convert_ids_to_tokens(i), TAGS[l] if l >= 0 else "-")
       for i, l in zip(enc["input_ids"][:20], encode(ex)["labels"][:20])])

## Part 4: The model
* **Encoder:** XLM-R (shared by all heads).
* **BIO head:** one tag (O/B/I) per word, which finds the sub-complaint boundaries.
* **Institution head:** applied to the averaged representation of each sub-complaint. It gives an **independent score (0–1) per institution** (sigmoid), which is what the confidence gate needs ("both ≥ t", "one ≥ t", "both < t").
* **Department heads (chained):** one head **per institution**. During training the correct institution's head is used; at prediction time the head of the **predicted** institution is used, so the department options depend on the institution chosen.

In [ ]:
class RoutingModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(DROPOUT)
        self.bio_head = nn.Linear(hidden, len(TAGS))
        self.inst_head = nn.Sequential(nn.Linear(hidden, hidden), nn.Tanh(), nn.Dropout(DROPOUT),
                                       nn.Linear(hidden, len(INSTITUTIONS)))
        self.dept_heads = nn.ModuleDict({
            inst: nn.Sequential(nn.Linear(hidden, hidden), nn.Tanh(), nn.Dropout(DROPOUT),
                                nn.Linear(hidden, len(DEPARTMENTS[inst])))
            for inst in INSTITUTIONS})

    def encode(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state

    @staticmethod
    def pool(hidden, spans):
        """spans: rows (batch_index, start, end) -> mean of the pieces in each span."""
        if len(spans) == 0:
            return hidden.new_zeros((0, hidden.size(-1)))
        return torch.stack([hidden[b, s:e].mean(0) for b, s, e in spans.tolist()])

    def forward(self, input_ids, attention_mask, spans):
        hidden = self.encode(input_ids, attention_mask)
        bio_logits = self.bio_head(self.dropout(hidden))
        reps = self.dropout(self.pool(hidden, spans[:, :3]))
        inst_logits = self.inst_head(reps)
        dept_logits = {inst: head(reps) for inst, head in self.dept_heads.items()}
        return bio_logits, inst_logits, dept_logits

bce = nn.BCEWithLogitsLoss()
ce = nn.CrossEntropyLoss(ignore_index=-100)

def compute_loss(model, batch):
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    bio_logits, inst_logits, dept_logits = model(batch["input_ids"], batch["attention_mask"], batch["spans"])
    loss_bio = ce(bio_logits.view(-1, len(TAGS)).float(), batch["labels"].view(-1))
    spans = batch["spans"]
    inst_target = torch.zeros_like(inst_logits, dtype=torch.float)
    inst_target[torch.arange(len(spans), device=DEVICE), spans[:, 3]] = 1.0
    loss_inst = bce(inst_logits.float(), inst_target)
    loss_dept, n = 0.0, 0
    for inst, logits in dept_logits.items():
        sel = spans[:, 3] == INST2ID[inst]
        if sel.any():
            target = torch.zeros((int(sel.sum()), logits.size(1)), device=DEVICE)
            target[torch.arange(target.size(0), device=DEVICE), spans[sel, 4]] = 1.0
            loss_dept = loss_dept + bce(logits[sel].float(), target) * int(sel.sum())
            n += int(sel.sum())
    loss_dept = loss_dept / max(n, 1)
    total = (LOSS_WEIGHTS["bio"] * loss_bio + LOSS_WEIGHTS["institution"] * loss_inst
             + LOSS_WEIGHTS["department"] * loss_dept)
    return total, {"bio": float(loss_bio), "institution": float(loss_inst), "department": float(loss_dept)}

model = RoutingModel(MODEL_NAME).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")

## Part 5: Prediction and metrics (used for validation during training and for the final test)
**Metrics (as in the proposal):**
* **Segmentation:** precision / recall / F1 of sub-complaint spans (a span counts only if its start and end are exactly right; punctuation at the edge of a span is ignored for every system), plus *segmentation accuracy* = share of complaints where **all** spans are exactly right.
* **Institution accuracy** and **department accuracy** on the correct sub-complaints (scored separately; department is reported both with the correct institution given and chained after the predicted institution).
* **Misrouting rate (end-to-end):** share of real sub-complaints that do **not** reach the correct institution **and** department. The model's own spans are used; a predicted span is matched to a real one when they overlap by at least 50%.

In [ ]:
def tags_to_spans(tags):
    """Word-level tags -> list of (start, end). An I without a B before it starts a new span."""
    spans, start = [], None
    for i, t in enumerate(tags):
        if t == "B":
            if start is not None:
                spans.append((start, i))
            start = i
        elif t == "I":
            if start is None:
                start = i
        else:
            if start is not None:
                spans.append((start, i))
            start = None
    if start is not None:
        spans.append((start, len(tags)))
    return spans

def gold_spans(record):
    return [(s["token_start"], s["token_end"]) for s in record["sub_complaints"]]

EDGE_PUNCT = set(".,!?;:\"'()[]{}…“”‘’|෴")

def trim_spans(spans, tokens):
    """For scoring only: punctuation at the edge of a span is ignored, so a full stop counted
    on either side of a boundary is not an error. Applied equally to every system."""
    out = []
    for a, b in spans:
        while a < b and tokens[a] in EDGE_PUNCT:
            a += 1
        while b > a and tokens[b - 1] in EDGE_PUNCT:
            b -= 1
        if a < b:
            out.append((a, b))
    return out

def word_spans_to_pieces(item, spans):
    out = []
    for (a, b) in spans:
        ws = [w for w in range(a, b) if w in item["first"]]
        out.append((item["first"][ws[0]], item["last"][ws[-1]] + 1) if ws else None)
    return out

@torch.no_grad()
def predict(records, items, extra_spans=None):
    """Runs the model once per record. Returns, per record: predicted word tags and spans,
    and institution/department scores for the predicted spans, the gold spans and any extra spans
    (used by the rule-based baseline)."""
    model.eval()
    results = []
    order = sorted(range(len(items)), key=lambda i: len(items[i]["input_ids"]))
    for start in range(0, len(order), EVAL_BATCH_SIZE):
        idx = order[start:start + EVAL_BATCH_SIZE]
        batch = collate([items[i] for i in idx])
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            hidden = model.encode(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
            bio = model.bio_head(hidden).float().argmax(-1).cpu().numpy()
        hidden = hidden.float()
        for row, i in enumerate(idx):
            item, rec = items[i], records[i]
            tags = ["O"] * item["n_words"]
            for w, pos in item["first"].items():
                tags[w] = TAGS[bio[row, pos]]
            p_spans = tags_to_spans(tags)
            span_sets = {"pred": p_spans, "gold": gold_spans(rec)}
            if extra_spans is not None:
                span_sets["rule"] = extra_spans[i]
            scored = {}
            for name, wspans in span_sets.items():
                pieces = word_spans_to_pieces(item, wspans)
                ok = [(k, p) for k, p in enumerate(pieces) if p is not None]
                reps = torch.stack([hidden[row, s:e].mean(0) for _, (s, e) in ok]) if ok else None
                rows = [None] * len(wspans)
                if reps is not None:
                    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                        inst_p = torch.sigmoid(model.inst_head(reps).float()).cpu().numpy()
                        dept_p = {inst: torch.sigmoid(h(reps).float()).cpu().numpy()
                                  for inst, h in model.dept_heads.items()}
                    for j, (k, _) in enumerate(ok):
                        rows[k] = {"inst": inst_p[j], "dept": {inst: dept_p[inst][j] for inst in INSTITUTIONS}}
                scored[name] = rows
            results.append((i, {"tags": tags, "spans": span_sets, "scores": scored}))
    results.sort(key=lambda x: x[0])
    return [r for _, r in results]

def label_of(score_row):
    """Highest institution, then highest department of that institution (no thresholds)."""
    if score_row is None:
        return None, None
    inst = INSTITUTIONS[int(np.argmax(score_row["inst"]))]
    dept = DEPARTMENTS[inst][int(np.argmax(score_row["dept"][inst]))]
    return inst, dept

def iou(a, b):
    inter = max(0, min(a[1], b[1]) - max(a[0], b[0]))
    union = max(a[1], b[1]) - min(a[0], b[0])
    return inter / union if union else 0.0

def match_spans(gold, pred):
    """Greedy one-to-one matching by overlap; returns {gold_index: pred_index}."""
    pairs = sorted(((iou(g, p), gi, pi) for gi, g in enumerate(gold) for pi, p in enumerate(pred)),
                   reverse=True)
    used_g, used_p, m = set(), set(), {}
    for score, gi, pi in pairs:
        if score < SPAN_MATCH_IOU or gi in used_g or pi in used_p:
            continue
        m[gi] = pi; used_g.add(gi); used_p.add(pi)
    return m

def evaluate(records, preds, span_key="pred"):
    """All metrics for one system. span_key = 'pred' (model) or 'rule' (rule-based splitter)."""
    tp = fp = fn = exact = 0
    inst_ok = dept_given_ok = dept_chained_ok = n_gold = 0
    routed_ok, extra_segments = [], 0
    for rec, p in zip(records, preds):
        g = trim_spans(gold_spans(rec), rec["tokens"])
        raw_ps = p["spans"][span_key]
        keep = [k for k, sp in enumerate(raw_ps) if trim_spans([sp], rec["tokens"])]
        ps = [trim_spans([raw_ps[k]], rec["tokens"])[0] for k in keep]
        gs, pset = set(g), set(ps)
        tp += len(gs & pset); fp += len(pset - gs); fn += len(gs - pset)
        exact += int(gs == pset)
        # classification on the correct (gold) spans
        for k, sub in enumerate(rec["sub_complaints"]):
            row = p["scores"]["gold"][k]
            n_gold += 1
            if row is None:
                continue
            inst_pred, dept_chained = label_of(row)
            inst_ok += inst_pred == sub["institution"]
            dept_given = DEPARTMENTS[sub["institution"]][int(np.argmax(row["dept"][sub["institution"]]))]
            dept_given_ok += dept_given == sub["department"]
            dept_chained_ok += (inst_pred == sub["institution"] and dept_chained == sub["department"])
        # end-to-end routing with the system's own spans
        m = match_spans(g, ps)
        extra_segments += len(ps) - len(m)
        for k, sub in enumerate(rec["sub_complaints"]):
            ok = False
            if k in m:
                inst, dept = label_of(p["scores"][span_key][keep[m[k]]])
                ok = inst == sub["institution"] and dept == sub["department"]
            routed_ok.append(ok)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec_ = tp / (tp + fn) if tp + fn else 0.0
    return {
        "seg_precision": prec, "seg_recall": rec_,
        "seg_f1": 2 * prec * rec_ / (prec + rec_) if prec + rec_ else 0.0,
        "seg_accuracy": exact / max(len(records), 1),
        "institution_accuracy": inst_ok / max(n_gold, 1),
        "department_accuracy_given_institution": dept_given_ok / max(n_gold, 1),
        "department_accuracy_chained": dept_chained_ok / max(n_gold, 1),
        "routing_accuracy": float(np.mean(routed_ok)) if routed_ok else 0.0,
        "misrouting_rate": 1 - float(np.mean(routed_ok)) if routed_ok else 1.0,
        "extra_segments": extra_segments,
        "n_records": len(records), "n_sub_complaints": len(routed_ok),
        "_routed_ok": routed_ok,
    }

def show(metrics, title=""):
    keys = ["seg_precision", "seg_recall", "seg_f1", "seg_accuracy", "institution_accuracy",
            "department_accuracy_given_institution", "department_accuracy_chained", "misrouting_rate"]
    print(title, " | ".join(f"{k}={metrics[k]:.4f}" for k in keys))

## Part 6: Training
The validation set is evaluated after every epoch. The model with the best validation score (average of segmentation F1 and routing accuracy) is saved to `outputs/model/best_model.pt`. Training stops early if validation stops improving.

In [ ]:
train_loader = DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
                          generator=torch.Generator().manual_seed(SEED))
head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
enc_decay = [p for n, p in model.encoder.named_parameters() if not any(x in n for x in ["bias", "LayerNorm"])]
enc_nodecay = [p for n, p in model.encoder.named_parameters() if any(x in n for x in ["bias", "LayerNorm"])]
optimizer = torch.optim.AdamW([
    {"params": enc_decay, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
    {"params": enc_nodecay, "lr": LEARNING_RATE, "weight_decay": 0.0},
    {"params": head_params, "lr": HEAD_LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
])
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

BEST_PATH = OUT_DIR / "model" / "best_model.pt"
history, best_score, bad_epochs = [], -1.0, 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0, sums, steps = time.time(), Counter(), 0
    for step, batch in enumerate(train_loader, 1):
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            loss, parts = compute_loss(model, batch)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        sums.update(parts); sums["total"] += float(loss); steps += 1
        if step % 100 == 0:
            print(f"  epoch {epoch} step {step}/{len(train_loader)} loss {sums['total'] / steps:.4f}")
    val_preds = predict(data["validation"], datasets["validation"].items)
    vm = evaluate(data["validation"], val_preds)
    score = (vm["seg_f1"] + vm["routing_accuracy"]) / 2
    row = {"epoch": epoch, "train_loss": sums["total"] / steps,
           **{f"train_loss_{k}": sums[k] / steps for k in ["bio", "institution", "department"]},
           **{f"val_{k}": v for k, v in vm.items() if not k.startswith("_")},
           "val_score": score, "minutes": (time.time() - t0) / 60}
    history.append(row)
    improved = score > best_score
    if improved:
        best_score, bad_epochs = score, 0
        torch.save(model.state_dict(), BEST_PATH)
    else:
        bad_epochs += 1
    print(f"Epoch {epoch}: train loss {row['train_loss']:.4f} | val seg F1 {vm['seg_f1']:.4f} | "
          f"val routing acc {vm['routing_accuracy']:.4f} | {row['minutes']:.1f} min"
          f"{'  <- best, saved' if improved else ''}")
    if bad_epochs >= EARLY_STOP_PATIENCE:
        print("Early stopping: validation did not improve.")
        break

pd.DataFrame(history).to_csv(OUT_DIR / "metrics" / "training_history.csv", index=False)
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
print(f"\nBest validation score {best_score:.4f}; best model reloaded.")

In [ ]:
# training curves
try:
    import matplotlib.pyplot as plt
    h = pd.DataFrame(history)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(h.epoch, h.train_loss, marker="o"); ax[0].set_title("Training loss"); ax[0].set_xlabel("epoch")
    for k, lab in [("val_seg_f1", "segmentation F1"), ("val_routing_accuracy", "routing accuracy"),
                   ("val_institution_accuracy", "institution acc."),
                   ("val_department_accuracy_chained", "department acc. (chained)")]:
        ax[1].plot(h.epoch, h[k], marker="o", label=lab)
    ax[1].set_title("Validation"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].set_ylim(0, 1.02)
    plt.tight_layout(); plt.savefig(OUT_DIR / "figures" / "training_curves.png", dpi=150); plt.show()
except Exception as e:
    print("Plot skipped:", e)

## Part 7: Confidence thresholds (calibrated on validation only)
The gates need a threshold `t`. Using the **validation** set (never the test set), the lowest threshold is chosen at which the predictions that pass the gate are at least `TARGET_GATE_ACCURACY` (95%) correct. Predictions below the threshold are flagged `needs_verification`. One threshold is chosen for institutions and one for departments.

In [ ]:
val_preds = predict(data["validation"], datasets["validation"].items)

def calibrate(rows, target):
    """rows: list of (max_score, correct). Returns chosen threshold and the full table."""
    table = []
    for t in np.round(np.arange(0.30, 0.96, 0.05), 2):
        passed = [ok for s, ok in rows if s >= t]
        table.append({"threshold": t, "coverage": len(passed) / max(len(rows), 1),
                      "accuracy_of_passed": float(np.mean(passed)) if passed else float("nan")})
    table = pd.DataFrame(table)
    ok = table[(table.accuracy_of_passed >= target)]
    chosen = float(ok.threshold.min()) if len(ok) else 0.5
    return max(chosen, 0.5), table

inst_rows, dept_rows = [], []
for rec, p in zip(data["validation"], val_preds):
    for k, sub in enumerate(rec["sub_complaints"]):
        row = p["scores"]["gold"][k]
        if row is None:
            continue
        inst_rows.append((float(row["inst"].max()), INSTITUTIONS[int(row["inst"].argmax())] == sub["institution"]))
        d = row["dept"][sub["institution"]]
        dept_rows.append((float(d.max()), DEPARTMENTS[sub["institution"]][int(d.argmax())] == sub["department"]))

T_INST, inst_table = calibrate(inst_rows, TARGET_GATE_ACCURACY)
T_DEPT, dept_table = calibrate(dept_rows, TARGET_GATE_ACCURACY)
inst_table.to_csv(OUT_DIR / "metrics" / "threshold_calibration_institution.csv", index=False)
dept_table.to_csv(OUT_DIR / "metrics" / "threshold_calibration_department.csv", index=False)
print("Institution threshold t =", T_INST); print(inst_table.to_string(index=False))
print("\nDepartment threshold t =", T_DEPT); print(dept_table.to_string(index=False))

# save everything needed to use the model later
tokenizer.save_pretrained(OUT_DIR / "model")
json.dump({"model_name": MODEL_NAME, "max_len": MAX_LEN, "institutions": INSTITUTIONS,
           "departments": DEPARTMENTS, "tags": TAGS, "threshold_institution": T_INST,
           "threshold_department": T_DEPT, "seed": SEED},
          open(OUT_DIR / "model" / "routing_config.json", "w"), indent=2, ensure_ascii=False)
print("\nSaved model, tokenizer and routing_config.json to", OUT_DIR / "model")

## Part 8: Baselines
**Baseline 1: TF-IDF single-label classifier** (CitiWatch-style). It is trained on the single-topic training complaints and gives **one** institution + department per complaint, as today's systems do. On a complaint with *k* topics it can route at most 1 of *k* sub-complaints correctly (the structural ceiling).

**Baseline 2: rule-based splitter.** It splits only at known conjunctions/connectors (the connectors seen in training plus common conjunctions), then uses **the same classification heads** as the model. This isolates learned segmentation from rule-based segmentation.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union

single_train = [r for r in data["train"] if r["complaint_type"] == "single"]
tfidf = make_pipeline(
    make_union(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
               TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=1, sublinear_tf=True,
                               token_pattern=r"\S+")),
    LogisticRegression(max_iter=3000, C=5.0, class_weight="balanced"))
tfidf.fit([r["raw_complaint_text"] for r in single_train],
          [f'{r["target_institutions"][0]}|{r["target_departments"][0]}' for r in single_train])

def evaluate_tfidf(records):
    labels = tfidf.predict([r["raw_complaint_text"] for r in records])
    routed_ok, inst_ok, dept_ok, n = [], 0, 0, 0
    for rec, lab in zip(records, labels):
        inst, dept = lab.split("|", 1)
        for sub in rec["sub_complaints"]:
            ok = inst == sub["institution"] and dept == sub["department"]
            routed_ok.append(ok)
        if rec["complaint_type"] == "single":
            n += 1; sub = rec["sub_complaints"][0]
            inst_ok += inst == sub["institution"]; dept_ok += ok
    return {"seg_precision": float("nan"), "seg_recall": float("nan"), "seg_f1": float("nan"),
            "seg_accuracy": float(np.mean([r["num_sub_complaints"] == 1 for r in records])),
            "institution_accuracy": inst_ok / n if n else float("nan"),
            "department_accuracy_given_institution": float("nan"),
            "department_accuracy_chained": dept_ok / n if n else float("nan"),
            "routing_accuracy": float(np.mean(routed_ok)), "misrouting_rate": 1 - float(np.mean(routed_ok)),
            "extra_segments": 0, "n_records": len(records), "n_sub_complaints": len(routed_ok),
            "_routed_ok": routed_ok}

# ---- rule-based splitter: the connectors used in training + common conjunctions
RULE_CONNECTORS = [
    "ඒ වගේම", "ඒ වාගේම", "එමෙන්ම", "තවද", "ඊට අමතරව", "ඒ හැර", "ඒත් එක්කම", "තවත් දෙයක් තමයි",
    "අනෙක් කාරණය නම්", "තව ප්‍රශ්නයක් තියෙනවා", "තවත් ගැටලුවක් තියෙනවා", "සහ", "හා", "නමුත්", "ඒත්",
    "also", "and also", "and", "in addition", "furthermore", "moreover", "apart from that", "besides that",
    "as well as that", "another issue", "there is another problem too", "but", "though", "although",
    "ewagema", "e wagema", "thawa", "ita amatharawa", "anith eka", "thawa prashnayak thiyenawa",
    "e witharak nemei",
]
PUNCT = set(".,!?;:\"'()[]{}…“”‘’|෴")
RULE_PATTERNS = sorted({tuple(unicodedata.normalize("NFC", c).lower().split()) for c in RULE_CONNECTORS},
                       key=len, reverse=True)

def rule_split(tokens):
    low = [unicodedata.normalize("NFC", t).lower() for t in tokens]
    spans, start, i = [], 0, 0
    while i < len(low):
        hit = next((p for p in RULE_PATTERNS if tuple(low[i:i + len(p)]) == p), None)
        if hit and i > start:
            end = i
            while end > start and low[end - 1] in {",", ";"}:
                end -= 1
            if end > start:
                spans.append((start, end))
            j = i + len(hit)
            while j < len(low) and low[j] in {",", ":", "."}:
                j += 1
            start = i = j
            continue
        i += 1
    if start < len(low):
        spans.append((start, len(low)))
    return [s for s in spans if any(t not in PUNCT for t in low[s[0]:s[1]])] or [(0, len(low))]

print("TF-IDF baseline trained on", len(single_train), "single-topic complaints.")
ex = next(r for r in data["validation"] if r["complaint_type"] == "two_cross")
print("\nRule splitter example:\n ", ex["raw_complaint_text"][:200])
print("  gold spans:", gold_spans(ex), " rule spans:", rule_split(ex["tokens"]))

## Part 9: Final evaluation on the TEST set
This is the only place the test set is used. Results are reported overall, per complaint type, and for test complaints with **seen vs unseen (test-only) connectors**.

In [ ]:
test = data["test"]
rule_spans = [rule_split(r["tokens"]) for r in test]
test_preds = predict(test, datasets["test"].items, extra_spans=rule_spans)

def by_group(records, preds, system):
    groups = {"all": list(range(len(records)))}
    for t in ["single", "two_same", "two_cross", "three"]:
        groups[t] = [i for i, r in enumerate(records) if r["complaint_type"] == t]
    groups["multi_topic"] = [i for i, r in enumerate(records) if r["complaint_type"] != "single"]
    groups["multi_seen_connector"] = [i for i in groups["multi_topic"] if not records[i].get("unseen_connector")]
    groups["multi_unseen_connector"] = [i for i in groups["multi_topic"] if records[i].get("unseen_connector")]
    out = {}
    for g, idx in groups.items():
        if not idx:
            continue
        recs = [records[i] for i in idx]
        if system == "tfidf":
            out[g] = evaluate_tfidf(recs)
        else:
            out[g] = evaluate(recs, [preds[i] for i in idx], span_key="pred" if system == "model" else "rule")
    return out

results = {"XLM-R + BIO (proposed)": by_group(test, test_preds, "model"),
           "Rule-based splitter + same heads": by_group(test, test_preds, "rule"),
           "TF-IDF single-label": by_group(test, test_preds, "tfidf")}

rows = []
for system, groups in results.items():
    for g, m in groups.items():
        rows.append({"system": system, "group": g, **{k: v for k, v in m.items() if not k.startswith("_")}})
table = pd.DataFrame(rows)
table.to_csv(OUT_DIR / "metrics" / "test_results_all_groups.csv", index=False)

cols = ["seg_f1", "seg_accuracy", "institution_accuracy", "department_accuracy_chained", "misrouting_rate"]
main = table[table.group.isin(["all", "single", "multi_topic", "two_same", "two_cross", "three"])]
pd.set_option("display.width", 200)
print("=== TEST RESULTS ===")
print(main.pivot(index="group", columns="system", values="misrouting_rate").round(4).to_string())
print("\nMain comparison (all test complaints):")
print(table[table.group == "all"].set_index("system")[cols].round(4).to_string())
print("\nSegmentation F1 by connector (multi-topic test complaints):")
print(table[table.group.isin(["multi_seen_connector", "multi_unseen_connector"])]
      .pivot(index="group", columns="system", values="seg_f1").round(4).to_string())

### Statistical comparison (McNemar's exact test)
Each real test sub-complaint is either routed correctly or not by each system, so the systems are compared pair-wise on the same sub-complaints. A p-value below 0.05 means the difference is statistically significant.

In [ ]:
def mcnemar(a_ok, b_ok):
    b = sum(1 for x, y in zip(a_ok, b_ok) if x and not y)
    c = sum(1 for x, y in zip(a_ok, b_ok) if y and not x)
    n = b + c
    if n == 0:
        return b, c, 1.0
    k = min(b, c)
    p = min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)
    return b, c, p

stats = []
proposed = results["XLM-R + BIO (proposed)"]
for base in ["Rule-based splitter + same heads", "TF-IDF single-label"]:
    for g in ["all", "multi_topic"]:
        b, c, p = mcnemar(proposed[g]["_routed_ok"], results[base][g]["_routed_ok"])
        stats.append({"compared_with": base, "group": g, "proposed_right_baseline_wrong": b,
                      "baseline_right_proposed_wrong": c, "p_value": p,
                      "significant_at_0.05": p < 0.05})
stats = pd.DataFrame(stats)
stats.to_csv(OUT_DIR / "metrics" / "significance_tests.csv", index=False)
print(stats.to_string(index=False))

### Structural ceiling of the single-label baseline
On a complaint with *k* topics, a single-label classifier can route at most 1/*k* of its sub-complaints correctly, however well it is trained.

In [ ]:
ceiling = []
for t, k in [("single", 1), ("two_same", 2), ("two_cross", 2), ("three", 3)]:
    m = results["TF-IDF single-label"].get(t)
    if m:
        ceiling.append({"type": t, "topics_k": k, "maximum_possible_routing_accuracy": 1 / k,
                        "tfidf_routing_accuracy": round(m["routing_accuracy"], 4),
                        "proposed_routing_accuracy": round(proposed[t]["routing_accuracy"], 4)})
ceiling = pd.DataFrame(ceiling)
ceiling.to_csv(OUT_DIR / "metrics" / "structural_ceiling.csv", index=False)
print(ceiling.to_string(index=False))

### Confusion matrices and error files
Confusion matrices are on the correct (gold) sub-complaints. `test_predictions.xlsx` lists every test complaint with the real and predicted sub-complaints; `test_errors.xlsx` has only the mistakes, for the qualitative error analysis.

In [ ]:
from sklearn.metrics import confusion_matrix

y_true_i, y_pred_i, y_true_d, y_pred_d = [], [], [], []
pred_rows, err_rows = [], []
for rec, p in zip(test, test_preds):
    for k, sub in enumerate(rec["sub_complaints"]):
        row = p["scores"]["gold"][k]
        if row is None:
            continue
        inst, dept = label_of(row)
        y_true_i.append(sub["institution"]); y_pred_i.append(inst)
        y_true_d.append(f'{sub["institution"]}|{sub["department"]}'); y_pred_d.append(f"{inst}|{dept}")
    toks = rec["tokens"]
    m = match_spans(gold_spans(rec), p["spans"]["pred"])
    g_trim = set(trim_spans(gold_spans(rec), toks)); p_trim = set(trim_spans(p["spans"]["pred"], toks))
    pred_desc = []
    for j, (a, b) in enumerate(p["spans"]["pred"]):
        inst, dept = label_of(p["scores"]["pred"][j])
        pred_desc.append(f"[{inst} / {dept}] " + " ".join(toks[a:b]))
    gold_desc = [f'[{s["institution"]} / {s["department"]}] {s["text"]}' for s in rec["sub_complaints"]]
    routed = []
    for k, s in enumerate(rec["sub_complaints"]):
        ok = False
        if k in m:
            inst, dept = label_of(p["scores"]["pred"][m[k]])
            ok = (inst, dept) == (s["institution"], s["department"])
        routed.append(ok)
    row = {"complaint_id": rec["complaint_id"], "type": rec["complaint_type"],
           "unseen_connector": rec.get("unseen_connector"), "text": rec["raw_complaint_text"],
           "real_sub_complaints": "\n".join(gold_desc), "predicted_sub_complaints": "\n".join(pred_desc),
           "segmentation_exact": g_trim == p_trim,
           "all_routed_correctly": all(routed)}
    pred_rows.append(row)
    if not row["all_routed_correctly"] or not row["segmentation_exact"]:
        err_rows.append(row)
pd.DataFrame(pred_rows).to_excel(OUT_DIR / "predictions" / "test_predictions.xlsx", index=False)
pd.DataFrame(err_rows).to_excel(OUT_DIR / "predictions" / "test_errors.xlsx", index=False)
print(f"{len(err_rows)} of {len(pred_rows)} test complaints have at least one error -> test_errors.xlsx")

labels_d = [f"{i}|{d}" for i in INSTITUTIONS for d in DEPARTMENTS[i]]
cm_i = pd.DataFrame(confusion_matrix(y_true_i, y_pred_i, labels=INSTITUTIONS), index=INSTITUTIONS, columns=INSTITUTIONS)
cm_d = pd.DataFrame(confusion_matrix(y_true_d, y_pred_d, labels=labels_d), index=labels_d, columns=labels_d)
cm_i.to_csv(OUT_DIR / "metrics" / "confusion_institution.csv"); cm_d.to_csv(OUT_DIR / "metrics" / "confusion_department.csv")
print("\nInstitution (rows = real, columns = predicted):\n", cm_i.to_string())
print("\nDepartment:\n", cm_d.to_string())
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 6.5))
    norm = cm_d.values / np.maximum(cm_d.values.sum(1, keepdims=True), 1)
    ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels_d))); ax.set_xticklabels(labels_d, rotation=45, ha="right")
    ax.set_yticks(range(len(labels_d))); ax.set_yticklabels(labels_d)
    for a in range(len(labels_d)):
        for b in range(len(labels_d)):
            ax.text(b, a, int(cm_d.values[a, b]), ha="center", va="center",
                    color="white" if norm[a, b] > 0.5 else "black", fontsize=9)
    ax.set_xlabel("predicted"); ax.set_ylabel("real"); ax.set_title("Department confusion matrix (test)")
    plt.tight_layout(); plt.savefig(OUT_DIR / "figures" / "confusion_department.png", dpi=150); plt.show()
except Exception as e:
    print("Plot skipped:", e)

json.dump({"thresholds": {"institution": T_INST, "department": T_DEPT},
           "test": {s: {g: {k: v for k, v in m.items() if not k.startswith("_")} for g, m in gr.items()}
                    for s, gr in results.items()}},
          open(OUT_DIR / "metrics" / "test_metrics.json", "w"), indent=2)
print("\nAll metrics saved to", OUT_DIR / "metrics")

## Part 10: The routing engine (confidence gates + `is_fake`) and PP1 demo
This is the component's real behaviour: text in → sub-complaints out, each routed or stopped according to the gates in the design:

**Gate 1 (institution):** both scores ≥ t → higher one; one ≥ t → that one; both < t → higher one **and** `needs_verification = "institution"`, stop.
If `is_fake = True` the sub-complaint **stops at the institution** (the officer checks real/fake).
**Gate 2 (department, only if gate 1 passed and is_fake = False):** highest department; if its score < t → `needs_verification = "department"`.
The full original complaint text and `is_fake` travel with every sub-complaint.

In [ ]:
TOKEN_PEEL = set(".,!?;:\"'()[]{}…“”‘’|෴")

def tokenize_words(text):
    """Exactly the same word tokeniser used to build the dataset."""
    tokens = []
    for chunk in " ".join(unicodedata.normalize("NFC", text).split()).split():
        lead, trail = [], []
        while chunk and chunk[0] in TOKEN_PEEL:
            lead.append(chunk[0]); chunk = chunk[1:]
        while chunk and chunk[-1] in TOKEN_PEEL:
            trail.insert(0, chunk[-1]); chunk = chunk[:-1]
        tokens.extend(lead)
        if chunk:
            tokens.append(chunk)
        tokens.extend(trail)
    return tokens

def mask_identifiers(text):
    """Same idea as the LECO cleaning: emails -> <EMAIL>, long or masked numbers -> <NUMBER>."""
    text = re.sub(r"<?[\w.+-]+@[\w-]+(\.[\w-]+)+>?", "<EMAIL>", text)
    text = re.sub(r"(?<![\d*])\+?(?=[\d*]*\d)(?=[\d*]*\*{2})[\d*]{4,}(?![\d*])", "<NUMBER>", text)
    text = re.sub(r"(?<![\w/.])\+?\d{2,}(?:[ -]\d{2,})*(?![\w/])",
                  lambda m: "<NUMBER>" if sum(c.isdigit() for c in m.group()) >= 7 else m.group(), text)
    return text

def route_complaint(text, is_fake=False, complaint_id=None):
    clean_text = mask_identifiers(text)
    tokens = tokenize_words(clean_text)
    rec = {"tokens": tokens, "bio_tags": ["O"] * len(tokens), "sub_complaints": []}
    item = encode(rec)
    p = predict([rec], [item])[0]
    subs = []
    for j, (a, b) in enumerate(p["spans"]["pred"]):
        row = p["scores"]["pred"][j]
        if row is None:
            continue
        s_inst = {inst: round(float(row["inst"][k]), 4) for k, inst in enumerate(INSTITUTIONS)}
        best_inst = max(s_inst, key=s_inst.get)
        out = {"sub_complaint_text": " ".join(tokens[a:b]), "institution_scores": s_inst,
               "is_fake": bool(is_fake), "full_complaint_text": text}
        passing = [i for i, s in s_inst.items() if s >= T_INST]
        if not passing:                                   # case 5 / 6
            out.update({"institution": best_inst, "department": None,
                        "needs_verification": "institution", "status": "stopped at institution",
                        "reason": "no institution score reached the threshold"})
        elif is_fake:                                     # cases 2 / 4 (and 6)
            out.update({"institution": max(passing, key=s_inst.get), "department": None,
                        "needs_verification": None, "status": "stopped at institution",
                        "reason": "is_fake = True: officer checks whether the complaint is real"})
        else:                                             # cases 1 / 3 -> department gate
            inst = max(passing, key=s_inst.get)
            d_scores = {d: round(float(row["dept"][inst][k]), 4) for k, d in enumerate(DEPARTMENTS[inst])}
            dept = max(d_scores, key=d_scores.get)
            low = d_scores[dept] < T_DEPT
            out.update({"institution": inst, "department": dept, "department_scores": d_scores,
                        "needs_verification": "department" if low else None,
                        "status": "routed to department (flagged)" if low else "routed to department",
                        "reason": "department score below threshold" if low else "confident"})
        subs.append(out)
    return {"complaint_id": complaint_id, "raw_complaint_text": text, "tokens": tokens,
            "bio_tags": p["tags"], "is_fake": bool(is_fake), "num_sub_complaints": len(subs),
            "sub_complaints": subs,
            "target_institutions": [s["institution"] for s in subs],
            "target_departments": [s["department"] for s in subs]}

# ---- PP1 demo: real test complaints (two_cross and three), with is_fake False and True
demo = []
for t in ["two_cross", "three", "two_same"]:
    rec = next(r for r in test if r["complaint_type"] == t)
    for fake in [False, True]:
        out = route_complaint(rec["raw_complaint_text"], is_fake=fake, complaint_id=rec["complaint_id"])
        demo.append(out)
        print("=" * 100)
        print(f'{rec["complaint_id"]} ({t}) | is_fake={fake}\n{rec["raw_complaint_text"]}\n')
        print("Expected:", list(zip(rec["target_institutions"], rec["target_departments"])))
        for s in out["sub_complaints"]:
            print(f'  -> {s["status"]:<32} institution={s["institution"]:<5} department={str(s["department"]):<20}'
                  f' needs_verification={s["needs_verification"]}\n     "{s["sub_complaint_text"][:90]}"')
json.dump(demo, open(OUT_DIR / "predictions" / "routing_demo.json", "w"), indent=2, ensure_ascii=False)

In [ ]:
# Try your own complaint (Sinhala, English or mixed):
my_complaint = "Monthly bill is too high this month. ඒ වගේම අපේ පාරේ කානු අවහිර වෙලා වතුර පිරිලා තියෙනවා."
print(json.dumps(route_complaint(my_complaint, is_fake=False), indent=2, ensure_ascii=False))

## Done. What is in `/kaggle/working/outputs/`
| Folder | Files |
|---|---|
| `model/` | `best_model.pt` (weights), tokenizer files, `routing_config.json` (labels + thresholds) |
| `metrics/` | `training_history.csv`, `test_results_all_groups.csv`, `test_metrics.json`, `significance_tests.csv`, `structural_ceiling.csv`, threshold tables, confusion matrices |
| `predictions/` | `test_predictions.xlsx`, `test_errors.xlsx` (for error analysis), `routing_demo.json` |
| `figures/` | `training_curves.png`, `confusion_department.png` |